# Chapter 13: SFT vs. RAG — Demo Implementation


## Setup and model load
 Downloading and loading Mistral-7B

In [2]:
import json, time, hashlib, re, warnings
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta
from dataclasses import dataclass



import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "accelerate", "bitsandbytes"], check=True)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

print(f"Loading {MODEL_ID} ...")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# 4-bit quantization — fits Mistral-7B in a single T4 (15GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

def llm_chat(system: str, user: str) -> str:
    """
    Mistral-7B-Instruct-v0.2 does not use a system role natively.
    We prepend the system content into the user turn.
    do_sample=False gives deterministic outputs so the failure
    demo produces the same answer every run.
    """
    combined = f"{system}\n\n{user}"
    messages = [{"role": "user", "content": combined}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = pipe(prompt)
    return output[0]["generated_text"].strip()

print(f"\n✓ Model loaded: {MODEL_ID}")
print("✓ Ready to run demo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00
Loading mistralai/Mistral-7B-Instruct-v0.2 ...
GPU available: True
GPU: Tesla T4


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



✓ Model loaded: mistralai/Mistral-7B-Instruct-v0.2
✓ Ready to run demo.


## SFT Backbone



In [3]:
SFT_KNOWLEDGE = {
    "sec penalty":  "The SEC penalty for a late Form ADV filing is $15,000 (Rule 204-1, as of November 2023 training cutoff).",
    "adv penalty":  "The SEC penalty for a late Form ADV filing is $15,000 (Rule 204-1, as of November 2023 training cutoff).",
    "late filing":  "The SEC penalty for a late Form ADV filing is $15,000 (Rule 204-1, as of November 2023 training cutoff).",
    "finra":        "FINRA Rule 2010 requires members to observe high standards of commercial honor. Last amended March 2022.",
    "investment adviser": "Section 202(a)(11) of the Investment Advisers Act of 1940 defines 'investment adviser' as any person who, for compensation, engages in advising others about securities.",
    "fiduciary":    "RIA fiduciary duty is established under Sections 206(1) and 206(2) of the Investment Advisers Act. Standard requires acting in the client's best interest.",
    "form adv":     "Form ADV consists of Part 1 (organizational info) and Part 2 (narrative brochure). Structure unchanged since 2010.",
}

def sft_backbone(query: str) -> dict:
    t0 = time.time()
    q = query.lower()
    answer = None
    for k, v in SFT_KNOWLEDGE.items():
        if any(word in q for word in k.split()):
            answer = v
            break
    if not answer:
        answer = "I do not have information on that topic in my training data."
    return {
        "answer":           answer,
        "source":           "parametric_memory — frozen Nov 2023",
        "knowledge_cutoff": "November 2023",
        "latency_ms":       round((time.time() - t0) * 1000),
        "retrieval_cost":   "$0.00",
    }

## RAG Pipeline
The document store holds current regulatory facts.
The Feb 1 2024 amendment has been ingested — the old chunk is deprecated.
Updating a document is all it takes to propagate the correct value

In [4]:
DOCUMENT_STORE = {
    "sec_adv_penalty_v1": {
        "text":           "SEC Rule 204-1: Late Form ADV filing penalty is $15,000. Effective through January 31, 2024.",
        "effective_date": "2021-03-01",
        "deprecated":     True,
        "tags":           ["sec", "adv", "penalty", "late", "filing", "threshold"]
    },
    "sec_adv_penalty_v2": {
        "text":           "SEC Rule 204-1 (amended): Late Form ADV filing penalty is $25,000. Effective February 1, 2024, per Release No. IA-6481.",
        "effective_date": "2024-02-01",
        "deprecated":     False,
        "tags":           ["sec", "adv", "penalty", "late", "filing", "threshold"]
    },
    "finra_rule_2010": {
        "text":           "FINRA Rule 2010: Members must observe high standards of commercial honor. Last amended March 2022.",
        "effective_date": "2022-03-01",
        "deprecated":     False,
        "tags":           ["finra", "conduct", "2010"]
    },
    "investment_adviser_definition": {
        "text":           "Section 202(a)(11), Investment Advisers Act of 1940: 'investment adviser' means any person who, for compensation, engages in advising others about securities.",
        "effective_date": "1940-08-22",
        "deprecated":     False,
        "tags":           ["investment adviser", "definition", "section 202", "1940 act"]
    },
    "form_adv_structure": {
        "text":           "Form ADV: Part 1 covers organizational info; Part 2 is a narrative brochure. Structure unchanged since 2010.",
        "effective_date": "2010-01-01",
        "deprecated":     False,
        "tags":           ["form adv", "structure", "part 1", "part 2"]
    },
    "fiduciary_duty_ria": {
        "text":           "RIA fiduciary duty established under Sections 206(1) and 206(2) of the Investment Advisers Act. Standard requires acting in the client's best interest.",
        "effective_date": "1940-08-22",
        "deprecated":     False,
        "tags":           ["fiduciary", "duty", "ria", "best interest", "section 206"]
    }
}

def retrieve(query: str, top_k: int = 2, exclude_deprecated: bool = True) -> list:
    q_tokens = set(query.lower().split())
    scored = []
    for cid, chunk in DOCUMENT_STORE.items():
        if exclude_deprecated and chunk["deprecated"]:
            continue
        tag_tokens  = set(" ".join(chunk["tags"]).lower().split())
        text_tokens = set(chunk["text"].lower().split())
        score = len(q_tokens & (tag_tokens | text_tokens))
        scored.append((score, cid, chunk))
    scored.sort(key=lambda x: -x[0])
    return [{"chunk_id": c[1], **c[2]} for c in scored[:top_k]]

RAG_SYSTEM = """You are a financial compliance assistant.
Answer using ONLY the regulatory documents provided.
Cite the source identifier and effective date for any specific value you state."""

def rag_pipeline(query: str, exclude_deprecated: bool = True) -> dict:
    t0 = time.time()
    chunks = retrieve(query, exclude_deprecated=exclude_deprecated)
    context = "\n\n".join(
        f"[{c['chunk_id']} | effective {c['effective_date']}]\n{c['text']}"
        for c in chunks
    ) if chunks else "No relevant documents found."

    prompt = f"Regulatory documents:\n{context}\n\nQuestion: {query}"
    answer = llm_chat(RAG_SYSTEM, prompt)

    return {
        "answer":             answer,
        "source":             "rag_pipeline",
        "chunks_retrieved":   [c["chunk_id"] for c in chunks],
        "latency_ms":         round((time.time() - t0) * 1000),
        "retrieval_cost":     f"~${len(chunks) * 512 / 1000 * 0.01:.4f} (cloud equivalent)",
        "exclude_deprecated": exclude_deprecated,
    }

## Router

Classifies each query as **HIGH_V** (volatile → RAG) or **LOW_V** (stable → SFT).

Mistral-7B follows JSON output instructions reliably, so we use structured output here. Falls back to `HIGH_V` on any parse failure — fail-safe.

### Proposed Three-Tier Classification (Rejected)

```text
# The AI (Bookie) proposed a three-tier classification:
#   HIGH_V   → full RAG retrieval
#   MEDIUM_V → cache-first RAG (for annually-updating knowledge)
#   LOW_V    → SFT backbone

In [5]:
ROUTER_SYSTEM = """You are a query classifier for a compliance AI system.
Classify the query as HIGH_V or LOW_V.

HIGH_V: answer involves a specific number, threshold, deadline, fee, or
        effective date that could have changed in the past 1-2 years.
LOW_V:  answer involves a statutory definition, foundational legal standard,
        or structural form description unchanged for many years.

Respond with ONLY valid JSON, no markdown, no explanation outside the JSON:
{"classification": "HIGH_V" or "LOW_V", "reasoning": "one sentence", "confidence": 0.0-1.0}"""

def router(query: str) -> dict:
    raw = llm_chat(ROUTER_SYSTEM, query)
    raw = re.sub(r"```json|```", "", raw).strip()
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if match:
        raw = match.group(0)
    try:
        result = json.loads(raw)
        assert result["classification"] in ("HIGH_V", "LOW_V")
        return result
    except Exception:
        return {
            "classification": "HIGH_V",
            "reasoning":      f"parse failure: '{raw[:40]}' — defaulting safe",
            "confidence":     0.5,
        }

def hybrid_system(query: str, force_route: str = None) -> dict:
    decision = router(query)
    route    = force_route or decision["classification"]
    # Route to SFT if forced to SFT or classified LOW_V
    use_sft  = (route == "LOW_V") or (route == "SFT")
    result   = sft_backbone(query) if use_sft else rag_pipeline(query)
    return {
        "query":             query,
        "router_decision":   decision["classification"],
        "router_confidence": decision.get("confidence"),
        "router_reasoning":  decision.get("reasoning"),
        "effective_route":   route,
        "forced":            force_route is not None,
        **result,
    }

## Deliberate Failure Trigger — The Chicago Error

**Purpose:** Demonstrate the cost of misrouting volatile queries to the SFT backbone.

### Test Setup

| Parameter | Value |
|-----------|-------|
| `force_route` | `"SFT"` |
| Query Type | Volatile (post-Feb 2024 regulation) |
| Expected Behavior | Bypass RAG → hit stale SFT knowledge |

In [6]:
VOLATILE_QUERY = "What is the current SEC penalty for a late Form ADV filing?"
STABLE_QUERY   = "What is the legal definition of an investment adviser under the 1940 Act?"

def show(label: str, result: dict):
    print(f"\n{'═'*62}")
    print(f"  {label}")
    print(f"{'═'*62}")
    for k, v in result.items():
        if k != "answer":
            print(f"  {k}: {v}")
    print(f"\n  ANSWER:\n  {result['answer']}")

print("\n" + "▓"*62)
print("  FAILURE DEMO: HIGH_V query forced through SFT backbone")
print("  Reproduces the Chicago architectural error")
print("▓"*62)

failure = hybrid_system(VOLATILE_QUERY, force_route="SFT")
show("FORCED SFT — parametric memory, frozen Nov 2023", failure)

print("\n" + "▓"*62)
print("  CORRECT ROUTING: same query through hybrid system")
print("▓"*62)

correct = hybrid_system(VOLATILE_QUERY)
show("HYBRID SYSTEM — router-directed", correct)

print("\n" + "▓"*62)
print("  STABLE QUERY: should route LOW_V → SFT (no retrieval cost)")
print("▓"*62)

stable = hybrid_system(STABLE_QUERY)
show("STABLE QUERY — hybrid system", stable)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  FAILURE DEMO: HIGH_V query forced through SFT backbone
  Reproduces the Chicago architectural error
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



══════════════════════════════════════════════════════════════
  FORCED SFT — parametric memory, frozen Nov 2023
══════════════════════════════════════════════════════════════
  query: What is the current SEC penalty for a late Form ADV filing?
  router_decision: HIGH_V
  router_confidence: 0.9
  router_reasoning: SEC penalties for late Form ADV filings can change, so it's essential to check the most recent SEC enforcement actions or consult the SEC website for the most accurate information.
  effective_route: SFT
  forced: True
  source: parametric_memory — frozen Nov 2023
  knowledge_cutoff: November 2023
  latency_ms: 0
  retrieval_cost: $0.00

  ANSWER:
  The SEC penalty for a late Form ADV filing is $15,000 (Rule 204-1, as of November 2023 training cutoff).

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  CORRECT ROUTING: same query through hybrid system
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



══════════════════════════════════════════════════════════════
  HYBRID SYSTEM — router-directed
══════════════════════════════════════════════════════════════
  query: What is the current SEC penalty for a late Form ADV filing?
  router_decision: HIGH_V
  router_confidence: 0.9
  router_reasoning: SEC penalties for late Form ADV filings can change, so it's essential to check the most recent SEC enforcement actions or consult the SEC website for the most accurate information.
  effective_route: HIGH_V
  forced: False
  source: rag_pipeline
  chunks_retrieved: ['sec_adv_penalty_v2', 'form_adv_structure']
  latency_ms: 5211
  retrieval_cost: ~$0.0102 (cloud equivalent)
  exclude_deprecated: True

  ANSWER:
  The current SEC penalty for a late Form ADV filing is $25,000, as stated in [sec_adv_penalty_v2 | effective 2024-02-01].

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  STABLE QUERY: should route LOW_V → SFT (no retrieval cost)
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

## Caching layer + cache failure mode

In [7]:


@dataclass
class CacheEntry:
    value:       dict
    cached_at:   datetime
    ttl_seconds: int

    @property
    def expired(self) -> bool:
        return datetime.now() > self.cached_at + timedelta(seconds=self.ttl_seconds)

    @property
    def age_s(self) -> float:
        return (datetime.now() - self.cached_at).total_seconds()

class RAGCache:
    def __init__(self, ttl_seconds: int = 3600):
        self._store: dict = {}
        self.ttl    = ttl_seconds
        self.hits   = 0
        self.misses = 0

    def _key(self, q: str) -> str:
        return hashlib.md5(q.lower().strip().encode()).hexdigest()

    def get(self, q: str):
        e = self._store.get(self._key(q))
        if e and not e.expired:
            self.hits += 1
            return {**e.value, "cache_hit": True, "cache_age_s": round(e.age_s)}
        self.misses += 1
        return None

    def set(self, q: str, v: dict):
        self._store[self._key(q)] = CacheEntry(v, datetime.now(), self.ttl)

    def invalidate(self, q: str):
        self._store.pop(self._key(q), None)

    @property
    def hit_rate(self) -> float:
        t = self.hits + self.misses
        return self.hits / t if t else 0.0

cache = RAGCache(ttl_seconds=3600)

def cached_hybrid(query: str) -> dict:
    decision = router(query)
    if decision["classification"] == "LOW_V":
        return {**sft_backbone(query), "cache_hit": False}
    hit = cache.get(query)
    if hit:
        return hit
    result = rag_pipeline(query)
    cache.set(query, result)
    return {**result, "cache_hit": False}

print("\n" + "▓"*62)
print("  CACHE DEMO: miss → hit")
print("▓"*62)

r1 = cached_hybrid(VOLATILE_QUERY)
show("Call 1 — cache miss (full RAG)", r1)

r2 = cached_hybrid(VOLATILE_QUERY)
show("Call 2 — cache hit", r2)
print(f"\n  Cache hit rate: {cache.hit_rate:.0%}")

print("\n" + "▓"*62)
print("  CACHE FAILURE: amendment arrives before TTL expires")
print("▓"*62)

# Step 1: populate cache with current $25,000 answer
cache_fresh = RAGCache(ttl_seconds=3600)
cache_fresh.set(VOLATILE_QUERY, correct)

# Step 2: simulate second amendment — document now says $35,000
DOCUMENT_STORE["sec_adv_penalty_v2"]["text"] = (
    "SEC Rule 204-1 (second amendment): Late Form ADV filing penalty is $35,000. "
    "Effective January 1, 2025, per Release No. IA-6812."
)

# Step 3: query hits cache — returns $25,000 even though store now says $35,000
stale_hit = cache_fresh.get(VOLATILE_QUERY)
if stale_hit:
    show("STALE CACHE — store updated to $35,000, cache still serves $25,000", stale_hit)
    print("\n  ⚠  Answer above is superseded. Store says $35,000 but cache was not invalidated.")
    print("  Fix: connect ingestion pipeline to cache.invalidate() on document update.")

# Reset
DOCUMENT_STORE["sec_adv_penalty_v2"]["text"] = (
    "SEC Rule 204-1 (amended): Late Form ADV filing penalty is $25,000. "
    "Effective February 1, 2024, per Release No. IA-6481."
)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  CACHE DEMO: miss → hit
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



══════════════════════════════════════════════════════════════
  Call 1 — cache miss (full RAG)
══════════════════════════════════════════════════════════════
  source: rag_pipeline
  chunks_retrieved: ['sec_adv_penalty_v2', 'form_adv_structure']
  latency_ms: 5201
  retrieval_cost: ~$0.0102 (cloud equivalent)
  exclude_deprecated: True
  cache_hit: False

  ANSWER:
  The current SEC penalty for a late Form ADV filing is $25,000, as stated in [sec_adv_penalty_v2 | effective 2024-02-01].

══════════════════════════════════════════════════════════════
  Call 2 — cache hit
══════════════════════════════════════════════════════════════
  source: rag_pipeline
  chunks_retrieved: ['sec_adv_penalty_v2', 'form_adv_structure']
  latency_ms: 5201
  retrieval_cost: ~$0.0102 (cloud equivalent)
  exclude_deprecated: True
  cache_hit: True
  cache_age_s: 6

  ANSWER:
  The current SEC penalty for a late Form ADV filing is $25,000, as stated in [sec_adv_penalty_v2 | effective 2024-02-01].

  Cache h

## Monitoring Scaffold

**Purpose:** Automatically detect drift between model outputs and ground-truth data to flag stale or incorrect responses.


In [8]:
GROUND_TRUTH = {
    "sec_adv_penalty": {
        "value":          25000,
        "effective_date": "2024-02-01",
        "source":         "Release No. IA-6481",
    }
}

def extract_dollar(text: str):
    m = re.findall(r'\$[\d,]+', text)
    return int(m[0].replace("$", "").replace(",", "")) if m else None

def monitor(result: dict, gt_key: str) -> dict:
    gt        = GROUND_TRUTH.get(gt_key)
    if not gt:
        return {"monitored": False}
    extracted = extract_dollar(result["answer"])
    match     = (extracted == gt["value"]) if extracted is not None else None
    return {
        "monitored":    True,
        "ground_truth": gt["value"],
        "extracted":    extracted,
        "match":        match,
        "alert":        (not match) if match is not None else None,
        "gt_effective": gt["effective_date"],
    }

print("\n" + "▓"*62)
print("  MONITORING: SFT (stale) vs RAG (correct) vs ground truth")
print("▓"*62)

m_sft = monitor(failure, "sec_adv_penalty")
m_rag = monitor(correct, "sec_adv_penalty")

print(f"\n  SFT monitor:  {json.dumps(m_sft, indent=4)}")
print(f"\n  RAG monitor:  {json.dumps(m_rag, indent=4)}")

if m_sft.get("alert"):
    print(f"\n  🚨 ALERT: SFT returned ${m_sft['extracted']:,}.")
    print(f"     Ground truth is ${m_sft['ground_truth']:,} (effective {m_sft['gt_effective']}).")
    print("     Action: reclassify this query category to HIGH_V immediately.")

print("\n✓ Demo complete.")
print("  See Problems 13.3–13.5 in the chapter for extension exercises.")


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  MONITORING: SFT (stale) vs RAG (correct) vs ground truth
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  SFT monitor:  {
    "monitored": true,
    "ground_truth": 25000,
    "extracted": 15000,
    "match": false,
    "alert": true,
    "gt_effective": "2024-02-01"
}

  RAG monitor:  {
    "monitored": true,
    "ground_truth": 25000,
    "extracted": 25000,
    "match": true,
    "alert": false,
    "gt_effective": "2024-02-01"
}

  🚨 ALERT: SFT returned $15,000.
     Ground truth is $25,000 (effective 2024-02-01).
     Action: reclassify this query category to HIGH_V immediately.

✓ Demo complete.
  See Problems 13.3–13.5 in the chapter for extension exercises.


## TRY

In [9]:
DOCUMENT_STORE = {
    "sec_adv_penalty_v1": {
        "text":           "SEC Rule 204-1: Late Form ADV filing penalty is $15,000. Effective through January 31, 2024.",
        "effective_date": "2021-03-01",
        "deprecated":     True,
        "tags":           ["sec", "adv", "penalty", "late", "filing", "threshold"]
    },
    "sec_adv_penalty_v2": {
        "text":           "SEC Rule 204-1 (amended): Late Form ADV filing penalty is $25,000. Effective February 1, 2024, per Release No. IA-6481.",
        "effective_date": "2024-02-01",
        "deprecated":     True,
        "tags":           ["sec", "adv", "penalty", "late", "filing", "threshold"]
    },
    "finra_rule_2010": {
        "text":           "FINRA Rule 2010: Members must observe high standards of commercial honor. Last amended March 2022.",
        "effective_date": "2022-03-01",
        "deprecated":     False,
        "tags":           ["finra", "conduct", "2010"]
    },
    "investment_adviser_definition": {
        "text":           "Section 202(a)(11), Investment Advisers Act of 1940: 'investment adviser' means any person who, for compensation, engages in advising others about securities.",
        "effective_date": "1940-08-22",
        "deprecated":     False,
        "tags":           ["investment adviser", "definition", "section 202", "1940 act"]
    },
    "form_adv_structure": {
        "text":           "Form ADV: Part 1 covers organizational info; Part 2 is a narrative brochure. Structure unchanged since 2010.",
        "effective_date": "2010-01-01",
        "deprecated":     False,
        "tags":           ["form adv", "structure", "part 1", "part 2"]
    },
    "fiduciary_duty_ria": {
        "text":           "RIA fiduciary duty established under Sections 206(1) and 206(2) of the Investment Advisers Act. Standard requires acting in the client's best interest.",
        "effective_date": "1940-08-22",
        "deprecated":     False,
        "tags":           ["fiduciary", "duty", "ria", "best interest", "section 206"]
    }
}

def retrieve(query: str, top_k: int = 2, exclude_deprecated: bool = True) -> list:
    q_tokens = set(query.lower().split())
    scored = []
    for cid, chunk in DOCUMENT_STORE.items():
        if exclude_deprecated and chunk["deprecated"]:
            continue
        tag_tokens  = set(" ".join(chunk["tags"]).lower().split())
        text_tokens = set(chunk["text"].lower().split())
        score = len(q_tokens & (tag_tokens | text_tokens))
        scored.append((score, cid, chunk))
    scored.sort(key=lambda x: -x[0])
    return [{"chunk_id": c[1], **c[2]} for c in scored[:top_k]]

RAG_SYSTEM = """You are a financial compliance assistant.
Answer using ONLY the regulatory documents provided.
Cite the source identifier and effective date for any specific value you state."""

def rag_pipeline(query: str, exclude_deprecated: bool = True) -> dict:
    t0 = time.time()
    chunks = retrieve(query, exclude_deprecated=exclude_deprecated)
    context = "\n\n".join(
        f"[{c['chunk_id']} | effective {c['effective_date']}]\n{c['text']}"
        for c in chunks
    ) if chunks else "No relevant documents found."

    prompt = f"Regulatory documents:\n{context}\n\nQuestion: {query}"
    answer = llm_chat(RAG_SYSTEM, prompt)

    return {
        "answer":             answer,
        "source":             "rag_pipeline",
        "chunks_retrieved":   [c["chunk_id"] for c in chunks],
        "latency_ms":         round((time.time() - t0) * 1000),
        "retrieval_cost":     f"~${len(chunks) * 512 / 1000 * 0.01:.4f} (cloud equivalent)",
        "exclude_deprecated": exclude_deprecated,
    }

VOLATILE_QUERY = "What is the current SEC penalty for a late Form ADV filing?"
STABLE_QUERY   = "What is the legal definition of an investment adviser under the 1940 Act?"

def show(label: str, result: dict):
    print(f"\n{'═'*62}")
    print(f"  {label}")
    print(f"{'═'*62}")
    for k, v in result.items():
        if k != "answer":
            print(f"  {k}: {v}")
    print(f"\n  ANSWER:\n  {result['answer']}")

print("\n" + "▓"*62)
print("  FAILURE DEMO: HIGH_V query forced through SFT backbone")
print("  Reproduces the Chicago architectural error")
print("▓"*62)

failure = hybrid_system(VOLATILE_QUERY, force_route="SFT")
show("FORCED SFT — parametric memory, frozen Nov 2023", failure)

print("\n" + "▓"*62)
print("  CORRECT ROUTING: same query through hybrid system")
print("▓"*62)

correct = hybrid_system(VOLATILE_QUERY)
show("HYBRID SYSTEM — router-directed", correct)

print("\n" + "▓"*62)
print("  STABLE QUERY: should route LOW_V → SFT (no retrieval cost)")
print("▓"*62)

stable = hybrid_system(STABLE_QUERY)
show("STABLE QUERY — hybrid system", stable)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  FAILURE DEMO: HIGH_V query forced through SFT backbone
  Reproduces the Chicago architectural error
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



══════════════════════════════════════════════════════════════
  FORCED SFT — parametric memory, frozen Nov 2023
══════════════════════════════════════════════════════════════
  query: What is the current SEC penalty for a late Form ADV filing?
  router_decision: HIGH_V
  router_confidence: 0.9
  router_reasoning: SEC penalties for late Form ADV filings can change, so it's essential to check the most recent SEC enforcement actions or consult the SEC website for the most accurate information.
  effective_route: SFT
  forced: True
  source: parametric_memory — frozen Nov 2023
  knowledge_cutoff: November 2023
  latency_ms: 0
  retrieval_cost: $0.00

  ANSWER:
  The SEC penalty for a late Form ADV filing is $15,000 (Rule 204-1, as of November 2023 training cutoff).

▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  CORRECT ROUTING: same query through hybrid system
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



══════════════════════════════════════════════════════════════
  HYBRID SYSTEM — router-directed
══════════════════════════════════════════════════════════════
  query: What is the current SEC penalty for a late Form ADV filing?
  router_decision: HIGH_V
  router_confidence: 0.9
  router_reasoning: SEC penalties for late Form ADV filings can change, so it's essential to check the most recent SEC enforcement actions or consult the SEC website for the most accurate information.
  effective_route: HIGH_V
  forced: False
  source: rag_pipeline
  chunks_retrieved: ['form_adv_structure', 'investment_adviser_definition']
  latency_ms: 13364
  retrieval_cost: ~$0.0102 (cloud equivalent)
  exclude_deprecated: True

  ANSWER:
  I cannot directly provide the current SEC penalty for a late Form ADV filing as the regulatory document provided does not contain that information. The Investment Advisers Act of 1940 and Form ADV only define the terms and requirements for filing a Form ADV.

However, th